# LOCAL DEV TEST-HDB Resale Flat Prices - Pipeline Orchestration

Deploys infrastructure, then runs every stage in order - each stage reads
the real metadata state, does its work, writes the real result back, and
proves it landed - before moving to the next:

```
ingestion_to_source -> raw_iceberg -> data_profiling -> cleaned_iceberg -> transformed_iceberg -> hashed_iceberg
```

One email alert either way: the first failure, or one final success summary.


In [ ]:
# import shutil
# import subprocess
# from pathlib import Path

# RUN_SETUP = True  # False to skip deployment and go straight to the pipeline run below
# SETUP_SCRIPT = Path("setup.sh").resolve()

# # Plain "bash" on PATH is unreliable from a Jupyter kernel on Windows: if WSL is
# # installed, C:\Windows\System32\bash.exe often shadows Git Bash's bash.exe on
# # the kernel's own PATH (independent of whatever terminal you launched Jupyter
# # from). WSL's bash then tries to interpret the Windows path below as a Linux
# # path and fails instantly, before running a single line of setup.sh - so check
# # well-known Git Bash locations first and only fall back to bare "bash".
# _GIT_BASH_CANDIDATES = [
#     r"C:\Program Files\Git\bin\bash.exe",
#     r"C:\Program Files (x86)\Git\bin\bash.exe",
# ]
# BASH_EXE = next((p for p in _GIT_BASH_CANDIDATES if Path(p).exists()), None) or shutil.which("bash") or "bash"

# if RUN_SETUP:
#     if not SETUP_SCRIPT.exists():
#         raise FileNotFoundError(
#             f"setup.sh not found at {SETUP_SCRIPT} - adjust the path above if you've moved "
#             "this notebook or setup.sh."
#         )
#     print(f"Deploying: running {SETUP_SCRIPT}")
#     print(f"Using bash: {BASH_EXE}\n")
#     result = subprocess.run(
#         [BASH_EXE, str(SETUP_SCRIPT)],
#         cwd=SETUP_SCRIPT.parent,
#         text=True
#     )
#     print("\nExit code:", result.returncode)
#     if result.returncode != 0:
#         raise RuntimeError(
#             f"setup.sh failed with exit code {result.returncode}"
#         )
#     print(
#         "\nDeploy completed: buckets/roles/database/topic verified, "
#         "latest scripts synced to S3."
#     )
# else:
#     print(
#         "Skipping setup.sh (RUN_SETUP=False) - assuming infrastructure "
#         "and scripts are already up to date."
#     )

## How each step is tracked

Every step - ingestion, raw, cleaned, transformed, hashed - goes through
the SAME 4 pieces, reused rather than repeated per step (all in
`pipeline-scripts/ETL/`):

- **call_context** (`metadata_lambda.create_context`) - reads the real
  metadata tables (`metadata_tables`, `table_parameters`,
  `table_watermarks`, `pipeline_runs`) before the step runs.
- **call_monitoring** (`orchestration.run_local_step` /
  `run_glue_job`) - actually runs the step and watches it to completion,
  success or failure.
- **call_lambda** (`metadata_lambda.call_metadata_lambda`) - the one AWS
  Lambda invoke underneath both create_context and update_context.
- **call_update_context** (`metadata_lambda.update_context`, via
  `context_tracking.run_step_with_context`) - writes the REAL result
  (success/failure, timing, record count) back to the real metadata
  tables, then re-reads fresh to verify it actually landed.

`context_tracking.run_step_with_context()` wraps all 4 into one call and
**passes the fresh context forward** to whatever runs next. Failed records
inside cleaned/transformed/hashed are routed to `failed_iceberg` by that
job itself (`route_to_failed()`), covered under that same step's context
update rather than a separate one. `orchestration.py` sends one SNS email
with the execution summary, and every step's outcome is logged as a real
row in `pipeline_runs` - success or failure, not just printed.


In [2]:
import sys

sys.path.insert(0, "pipeline-scripts/ETL")

from metadata_lambda import create_context
from context_tracking import run_step_with_context
from orchestration import run_pipeline

RUN_MODE = "local"          # "local" | "glue" - glue mode needs 6 AWS Glue Jobs that dont exist yet
INCLUDE_HASHED_STEP = True  # False to stop before job_5 (hashed_iceberg)


ModuleNotFoundError: No module named 'boto3'

In [ ]:
# call_context - read the real metadata tables right now
context = create_context()


In [ ]:
# call_lambda + call_update_context, demoed once on ingestion:
# run_step_with_context() runs the real job, writes the real result back to
# the real metadata tables via the Lambda, verifies it landed, then hands
# back the fresh context - passed forward into `context` for later use.
import job_1_ingestion_to_source as ingestion_job

result = run_step_with_context(ingestion_job.main, layer="ingestion_to_source")
context = result["context"]  # pass context forward


## Run the chain

The same call_context / call_monitoring / call_lambda / call_update_context
pattern above repeats automatically for raw_iceberg, data_profiling,
cleaned_iceberg, transformed_iceberg, and hashed_iceberg - `orchestration.py`
calls `run_step_with_context()` for every step, not just ingestion. Stops at
the first failure and emails that failure immediately; emails one success
summary if every step passes.


In [ ]:
pipeline_succeeded, run_log = run_pipeline(run_mode=RUN_MODE, include_hashed_step=INCLUDE_HASHED_STEP)


In [ ]:
# Log success - explicit pass/fail summary for this run
if pipeline_succeeded:
    print("PIPELINE SUCCEEDED - every step passed, success email sent.")
else:
    print("PIPELINE FAILED - see the step above marked FAILED; failure email already sent.")

for step_name, job_name, state in run_log:
    print(f"  {step_name:22s} ({job_name:30s}) -> {state}")


In [ ]:
# Verify - fresh call_context read, proving the real metadata tables (not
# just this notebook's memory) reflect everything the run above just did.
final_context = create_context()

print("\nLatest pipeline_runs:")
for run in final_context["tables"].get("pipeline_runs", [])[-len(run_log):]:
    print(" ", run)

print("\nCurrent table_watermarks:")
for wm in final_context["tables"].get("table_watermarks", []):
    print(" ", wm)
